# Step 1: Feature Engineering

Run `prepare_dataset.py` → `compute_features.py`

In [1]:
import os
import sys
from pathlib import Path

# HARDCODED to Copy workspace due to space in path name
PROJECT_ROOT = Path("/media/przem/linux_data/RiskYieldMM (Copy)")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Workspace not found: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working dir: {os.getcwd()}")
print(f"✓ Workspace: {PROJECT_ROOT.name}")

Working dir: /media/przem/linux_data/RiskYieldMM (Copy)
✓ Workspace: RiskYieldMM (Copy)


In [2]:
# Step 1a: Merge raw data sources → merged_8h_raw.parquet
from scripts.feature_engineering.prepare_dataset import merge_all_sources, print_dataset_summary

data_dir = PROJECT_ROOT / "fetchingByBit"
df_raw = merge_all_sources(data_dir)
print_dataset_summary(df_raw)

# Save
output_path = PROJECT_ROOT / "data" / "merged_8h_raw.parquet"
df_raw.to_parquet(output_path)
print(f"\n✓ Saved: {output_path}")

Loading data sources...
  OHLCV: 5,438 rows
  Mark Price: 5,438 rows
  Index Price: 5,438 rows
  Premium: 5,438 rows
  Open Interest: 5,438 rows
  Funding Rate: 5,438 rows
  Long/Short Ratio: 5,002 rows

Merging data sources...

MERGED DATASET SUMMARY

Total rows: 5,438
Date range: 2021-01-01 00:00:00+00:00 to 2025-12-18 08:00:00+00:00

Total RAW columns: 24
  Price (P): 12
  Liquidity (L): 2
  Liquidity+Sentiment (L_S): 1
  Funding (F): 1
  Derivative/Premium (D): 4
  Sentiment (S): 3
  Temporal (TM): 1

NaN counts (top 10):
  RAW_S_buyRatio_bnd_N: 444 (8.2%)
  RAW_S_sellRatio_bnd_N: 444 (8.2%)
  RAW_S_longShortRatio_rat_N: 444 (8.2%)

Column list:
  RAW_D_F_S_premiumClose_pct_N
  RAW_D_F_S_premiumHigh_pct_N
  RAW_D_F_S_premiumLow_pct_N
  RAW_D_F_S_premiumOpen_pct_N
  RAW_F_I_S_fundingRate_pct_N
  RAW_L_S_openInterest_abs_NN
  RAW_L_turnover_abs_NN
  RAW_L_volume_abs_NN
  RAW_P_close_abs_NN
  RAW_P_high_abs_NN
  RAW_P_indexClose_abs_NN
  RAW_P_indexHigh_abs_NN
  RAW_P_indexLow_abs_NN


In [3]:
# Step 1b: Compute features → features_8h.parquet
from scripts.feature_engineering.compute_features import compute_all_features, load_raw_data

# Load raw data (compute_features expects its own format)
df_for_features = load_raw_data(data_dir)
print(f"Loaded raw data: {df_for_features.shape}")

# Compute all features
features_df = compute_all_features(df_for_features)
print(f"Computed features: {features_df.shape}")

# Save
features_path = PROJECT_ROOT / "data" / "features_8h.parquet"
features_df.to_parquet(features_path)
print(f"\n✓ Saved: {features_path}")

Loaded raw data: (5438, 17)


/media/przem/linux_data/RiskYieldMM (Copy)/scripts/feature_engineering/compute_features.py:1319: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"M_N_V_rangeExpansion_{n}_rat_N"] = compute_range_expansion(df, n)
/media/przem/linux_data/RiskYieldMM (Copy)/scripts/feature_engineering/compute_features.py:1320: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"V_volOfVol_{n}_pct_N"] = compute_vol_of_vol(df, n)
/media/przem/linux_data/RiskYieldMM (Copy)/scripts/feature_engineering/compute_features.py:1322: PerformanceW

Computed features: (5438, 167)

✓ Saved: /media/przem/linux_data/RiskYieldMM (Copy)/data/features_8h.parquet


/media/przem/linux_data/RiskYieldMM (Copy)/scripts/feature_engineering/compute_features.py:1424: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"V_hurstExponent_{n}_rat_N"] = compute_hurst_exponent(df, n)
/media/przem/linux_data/RiskYieldMM (Copy)/scripts/feature_engineering/compute_features.py:1428: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"V_yangZhang_{n}_pct_N"] = compute_yang_zhang_volatility(df, n)
/media/przem/linux_data/RiskYieldMM (Copy)/scripts/feature_engineering/compute_features.py:1428: Perfor

In [4]:
# Validation summary
import pandas as pd

features = pd.read_parquet(PROJECT_ROOT / "data" / "features_8h.parquet")

print("=" * 60)
print("STEP 1 COMPLETE: FEATURE ENGINEERING")
print("=" * 60)
print(f"\n✓ features_8h.parquet")
print(f"  Shape: {features.shape}")
print(f"  Columns: {features.shape[1]}")
print(f"  Rows: {features.shape[0]:,}")

# Verify volMomentum fix
vol_cols = ["V_volMomentum_6_pct_N", "V_volMomentum_12_pct_N", "V_volMomentum_21_pct_N"]
print(f"\n✓ volMomentum clipping verified:")
for col in vol_cols:
    print(f"  {col}: max={features[col].max():.1f} (clipped at 15.0)")

print(f"\n✓ Ready for Step 2: Target generation")

STEP 1 COMPLETE: FEATURE ENGINEERING

✓ features_8h.parquet
  Shape: (5438, 167)
  Columns: 167
  Rows: 5,438

✓ volMomentum clipping verified:
  V_volMomentum_6_pct_N: max=15.0 (clipped at 15.0)
  V_volMomentum_12_pct_N: max=15.0 (clipped at 15.0)
  V_volMomentum_21_pct_N: max=15.0 (clipped at 15.0)

✓ Ready for Step 2: Target generation


# Step 2: Target Generation

Generate `analysis_8h.parquet` with targets:
- `y_direction_1bar`: Binary (1=up, 0=down)
- `y_forward_return_{1,3,6,12}`: Multi-horizon returns
- `y_volatility`: |forward_return_1|
- `y_vol_regime`: LOW/MED/HIGH (0/1/2)
- `y_trend_regime`: SMA crossover

In [5]:
# Step 2: Create analysis dataset with targets
from scripts.analysis.data import create_analysis_dataset

df_analysis = create_analysis_dataset()

print(f"\n✓ Saved: {PROJECT_ROOT / 'data' / 'analysis_8h.parquet'}")

Loading data...
Features: (5438, 167)
Raw: (5438, 24)
✓ Timestamps aligned

Computing forward returns...
  y_forward_return_1 (8h): mean=0.0357%, std=1.7614%
  y_forward_return_3 (24h): mean=0.1081%, std=3.0973%
  y_forward_return_6 (48h): mean=0.2088%, std=4.2931%
  y_forward_return_12 (96h): mean=0.4093%, std=6.0647%
  Volatility thresholds (from first 1000 bars): 25%=0.017272, 75%=0.029226

Saving to /media/przem/linux_data/RiskYieldMM (Copy)/data/analysis_8h.parquet...
✓ Shape: (5438, 178), Size: 6.32 MB

✓ Saved: /media/przem/linux_data/RiskYieldMM (Copy)/data/analysis_8h.parquet


In [6]:
# Verify Step 2 output
import polars as pl

df_check = pl.read_parquet(PROJECT_ROOT / "data" / "analysis_8h.parquet")

print("=" * 60)
print("STEP 2 COMPLETE: TARGET GENERATION")
print("=" * 60)
print(f"\n✓ analysis_8h.parquet")
print(f"  Shape: {df_check.shape}")

# Count target columns
target_cols = [c for c in df_check.columns if c.startswith('y_')]
print(f"  Target columns: {len(target_cols)}")
print(f"    {', '.join(target_cols)}")

print(f"\n✓ Ready for Step 3: Dataset generation")

STEP 2 COMPLETE: TARGET GENERATION

✓ analysis_8h.parquet
  Shape: (5438, 178)
  Target columns: 9
    y_forward_return_1, y_forward_return_3, y_forward_return_6, y_forward_return_12, y_direction, y_volatility, y_direction_strength, y_trend_regime, y_vol_regime

✓ Ready for Step 3: Dataset generation


# Step 3: Feature Optimization

Per-target feature optimization using optimization pipeline.

**Why per-target optimization?**
- InteractionOptimizer selects feature pairs by IC against the TARGET
- Features predictive of `direction` ≠ features predictive of `volatility`
- RollingZScore HELPS direction/returns, HURTS volatility/regime targets

**Pipeline by target type:**

| Target        | Pipeline Steps                           |
|---------------|------------------------------------------|
| direction     | Winsorize → ExpandingRank → Interactions |
| returns       | Winsorize → RollingZScore → Interactions |
| volatility    | Winsorize only (base features are excellent) |
| vol_regime    | Winsorize → ExpandingRank → Interactions |
| trend_regime  | Winsorize → ExpandingRank → Interactions |

**Output:** 20 files `features_8h_optimized_{target}_{horizon}bar.parquet`

In [7]:
# Step 3: Run auto-optimization with THREADED execution (live output in Jupyter)
# Each worker processes one target-horizon combo SEQUENTIALLY (causal-safe)
#
# CAUSALITY GUARANTEE:
#   - Workers are isolated (no shared state)
#   - Each worker computes row-by-row internally
#   - Equivalent to sequential, just faster

from scripts.analysis.parallel_optimize import parallel_auto_optimize
from scripts.analysis.optimizers.expanding_rank_fast import HAS_NUMBA
import multiprocessing as mp
import importlib
import scripts.analysis.parallel_optimize as po
importlib.reload(po)  # Reload to get latest changes

print(f"Numba JIT available: {HAS_NUMBA}")
print(f"CPU cores available: {mp.cpu_count()}")
print("Running THREADED auto-optimization (live output, causal row-by-row preserved)\n")

# use_threading=True for Jupyter live output (default)
# use_threading=False for max speed (but no live output)
optimization_results = po.parallel_auto_optimize(
    horizons=[1, 3, 6, 12],
    targets=['direction', 'returns', 'volatility', 'vol_regime', 'trend_regime'],
    n_workers=4,
    save=True,
    use_threading=True  # Threads show live output in Jupyter (processes don't)
)

Numba JIT available: True
CPU cores available: 12
Running THREADED auto-optimization (live output, causal row-by-row preserved)

AUTO-OPTIMIZATION (THREADED, Causal-Safe)
Targets: ['direction', 'returns', 'volatility', 'vol_regime', 'trend_regime']
Horizons: [1, 3, 6, 12]
Total combinations: 20
Workers: 4
--------------------------------------------------------------------------------
  [ 1/20] direction_12bar: winsorize_zscore_interactions (IC=0.0225)
  [ 2/20] direction_1bar: winsorize_rank_interactions (IC=0.0129)
  [ 3/20] direction_3bar: winsorize_zscore_interactions (IC=0.0189)
  [ 4/20] direction_6bar: winsorize_rank_interactions (IC=0.0157)
  [ 5/20] returns_1bar: winsorize_interactions (IC=0.0231)
  [ 6/20] returns_3bar: winsorize_zscore_interactions (IC=0.0145)
  [ 7/20] returns_6bar: winsorize_zscore_interactions (IC=0.0171)
  [ 8/20] returns_12bar: winsorize_zscore_interactions (IC=0.0225)
  [ 9/20] volatility_1bar: winsorize_interactions (IC=0.0906)
  [10/20] volatility_3b

In [8]:
# VALIDATION: Verify NO FUTURE PEEKING in ExpandingRank
# This test proves row N only sees data from rows 0..N-1

import numpy as np
import pandas as pd
from scripts.analysis.optimizers.expanding_rank_fast import ExpandingRankOptimizerFast

print("=" * 70)
print("CAUSALITY VALIDATION: No Future Peeking Test")
print("=" * 70)

# Create test data with KNOWN pattern
np.random.seed(42)
n = 500
test_values = np.random.randn(n).cumsum()  # Random walk

# Add a SPIKE at row 400 that would be obvious if leaked
test_values[400:] += 100  # Huge jump

df_test = pd.DataFrame({'feature': test_values})

# Apply expanding rank
optimizer = ExpandingRankOptimizerFast(min_periods=50, use_numba=True)
optimizer.fit(df_test)
df_ranked = optimizer.transform(df_test)

# CAUSALITY TEST 1: Row 399 should NOT know about the spike at row 400
# If causal: rank[399] computed from rows 0..398 only (no spike knowledge)
# If leaky: rank[399] would be very low because future values are much higher

rank_before_spike = df_ranked['feature'].iloc[399]
rank_after_spike = df_ranked['feature'].iloc[400]

print(f"\n1. Spike Detection Test:")
print(f"   Row 399 (just before spike): rank = {rank_before_spike:.4f}")
print(f"   Row 400 (spike row): rank = {rank_after_spike:.4f}")

# Before spike: rank should be HIGH (near 1.0) - it's the cumsum maximum so far
# After spike: rank should be HIGH (near 1.0) - spike is much higher than history
if rank_before_spike > 0.9:
    print(f"   ✓ PASS: Row 399 has high rank ({rank_before_spike:.2f}) - doesn't know about future spike")
else:
    print(f"   ✗ FAIL: Row 399 has LOW rank ({rank_before_spike:.2f}) - FUTURE LEAKAGE DETECTED!")

# CAUSALITY TEST 2: Manually verify row 100
row_idx = 100
manual_hist = test_values[:row_idx]  # Rows 0..99
current_val = test_values[row_idx]
manual_rank = np.mean(manual_hist <= current_val)
computed_rank = df_ranked['feature'].iloc[row_idx]

print(f"\n2. Manual Verification (row {row_idx}):")
print(f"   Current value: {current_val:.4f}")
print(f"   History size: {len(manual_hist)} rows")
print(f"   Manual rank: {manual_rank:.6f}")
print(f"   Computed rank: {computed_rank:.6f}")
print(f"   Match: {'✓ PASS' if abs(manual_rank - computed_rank) < 1e-6 else '✗ FAIL'}")

# CAUSALITY TEST 3: Check expanding window behavior
print(f"\n3. Expanding Window Test:")
print(f"   First valid rank at row {optimizer.min_periods} (min_periods={optimizer.min_periods})")
print(f"   Rows 0-{optimizer.min_periods-1} should be NaN:")
nan_count = df_ranked['feature'].iloc[:optimizer.min_periods].isna().sum()
print(f"   NaN count in first {optimizer.min_periods} rows: {nan_count}")
print(f"   {'✓ PASS' if nan_count == optimizer.min_periods else '✗ FAIL'}")

print("\n" + "=" * 70)
if rank_before_spike > 0.9 and abs(manual_rank - computed_rank) < 1e-6:
    print("✓ ALL CAUSALITY TESTS PASSED - No future peeking detected")
else:
    print("✗ CAUSALITY VIOLATION DETECTED - Check implementation!")
print("=" * 70)

CAUSALITY VALIDATION: No Future Peeking Test

1. Spike Detection Test:
   Row 399 (just before spike): rank = 0.9950
   Row 400 (spike row): rank = 1.0000
   ✓ PASS: Row 399 has high rank (0.99) - doesn't know about future spike

2. Manual Verification (row 100):
   Current value: -11.8000
   History size: 100 rows
   Manual rank: 0.020000
   Computed rank: 0.020000
   Match: ✓ PASS

3. Expanding Window Test:
   First valid rank at row 50 (min_periods=50)
   Rows 0-49 should be NaN:
   NaN count in first 50 rows: 50
   ✓ PASS

✓ ALL CAUSALITY TESTS PASSED - No future peeking detected


In [9]:
# Verify Step 3: Check optimized feature files exist
from pathlib import Path

print("=" * 60)
print("STEP 3 VERIFICATION: OPTIMIZED FEATURES")
print("=" * 60)

data_dir = PROJECT_ROOT / "data"
targets = ['direction', 'returns', 'volatility', 'vol_regime', 'trend_regime']
horizons = [1, 3, 6, 12]

created_files = []
missing_files = []

for target in targets:
    for horizon in horizons:
        filename = f"features_8h_optimized_{target}_{horizon}bar.parquet"
        filepath = data_dir / filename
        if filepath.exists():
            size_mb = filepath.stat().st_size / (1024 * 1024)
            created_files.append((filename, size_mb))
        else:
            missing_files.append(filename)

print(f"\n✓ Created files: {len(created_files)}/20")
for f, size in created_files[:5]:  # Show first 5
    print(f"  {f}: {size:.2f} MB")
if len(created_files) > 5:
    print(f"  ... and {len(created_files) - 5} more")

if missing_files:
    print(f"\n✗ Missing files: {len(missing_files)}")
    for f in missing_files[:5]:
        print(f"  {f}")
else:
    print("\n✓ All 20 optimized feature files created")

print(f"\n✓ Ready for Step 4: Dataset generation")

STEP 3 VERIFICATION: OPTIMIZED FEATURES

✓ Created files: 20/20
  features_8h_optimized_direction_1bar.parquet: 7.61 MB
  features_8h_optimized_direction_3bar.parquet: 8.00 MB
  features_8h_optimized_direction_6bar.parquet: 7.70 MB
  features_8h_optimized_direction_12bar.parquet: 8.00 MB
  features_8h_optimized_returns_1bar.parquet: 7.92 MB
  ... and 15 more

✓ All 20 optimized feature files created

✓ Ready for Step 4: Dataset generation


# Step 4: Build Dataset Matrix

Generate 20 final datasets (5 targets × 4 horizons).

Each dataset contains:
- 166 base features (shared across all)
- 3-5 target-specific interaction features (from Step 3 optimization)
- 1 target column

**Target types:**
- `direction`: Binary (1=up, 0=down) — classification
- `returns`: Continuous forward return — regression
- `volatility`: |return| — risk/position sizing
- `vol_regime`: LOW/MED/HIGH — regime-aware strategies  
- `trend_regime`: Up/Down trend — trend-following

**Horizons:**
- `1bar` (8h): Intraday
- `3bar` (24h): Daily rebalancing  
- `6bar` (48h): Swing trading
- `12bar` (96h): Position trading

**Output:** `data/datasets/{target_type}_{horizon}bar.parquet`

In [10]:
# Step 4: Build the 20-dataset matrix
from scripts.analysis.run import cmd_build_datasets

# Build all 20 datasets: 5 targets × 4 horizons
# Each dataset:
#   - 166 base features (shared)
#   - 3-5 interaction features (target-specific from Step 3)
#   - 1 target column (y_{target_type})

print("Building 20-dataset matrix...")
print("This loads optimized features from Step 3 for each target-horizon combination\n")

dataset_summary = cmd_build_datasets(
    horizons=[1, 3, 6, 12],
    target_types=['direction', 'returns', 'volatility', 'vol_regime', 'trend_regime']
)

Building 20-dataset matrix...
This loads optimized features from Step 3 for each target-horizon combination

BUILDING DATASET MATRIX
Horizons: [1, 3, 6, 12]
Target types: ['direction', 'returns', 'volatility', 'vol_regime', 'trend_regime']
Total datasets: 20

Base features: 166
Rows: 5438

[direction/1-bar] Using 2 interactions from features_8h_optimized_direction_1bar.parquet
  ✓ direction_1bar.parquet: 168 features, 5437 valid targets

[returns/1-bar] Using 5 interactions from features_8h_optimized_returns_1bar.parquet
  ✓ returns_1bar.parquet: 171 features, 5437 valid targets

[volatility/1-bar] Using 5 interactions from features_8h_optimized_volatility_1bar.parquet
  ✓ volatility_1bar.parquet: 171 features, 5437 valid targets

[vol_regime/1-bar] Using 5 interactions from features_8h_optimized_vol_regime_1bar.parquet
  ✓ vol_regime_1bar.parquet: 171 features, 5438 valid targets

[trend_regime/1-bar] Using 4 interactions from features_8h_optimized_trend_regime_1bar.parquet
  ✓ trend_

In [11]:
# Final verification: All datasets created
import polars as pl

print("=" * 60)
print("STEP 4 COMPLETE: DATASET MATRIX")
print("=" * 60)

datasets_dir = PROJECT_ROOT / "data" / "datasets"
targets = ['direction', 'returns', 'volatility', 'vol_regime', 'trend_regime']
horizons = [1, 3, 6, 12]

print(f"\nDatasets directory: {datasets_dir}")
print("-" * 60)

total_valid = 0
for target in targets:
    for horizon in horizons:
        filepath = datasets_dir / f"{target}_{horizon}bar.parquet"
        if filepath.exists():
            df = pl.read_parquet(filepath)
            n_features = len([c for c in df.columns if not c.startswith(('y_', 'timestamp'))])
            n_interactions = len([c for c in df.columns if '×' in c])
            target_col = [c for c in df.columns if c.startswith('y_')][0]
            n_valid = df[target_col].drop_nulls().len()
            total_valid += n_valid
            print(f"✓ {target}_{horizon}bar: {n_features} base + {n_interactions} interactions, {n_valid:,} valid")
        else:
            print(f"✗ {target}_{horizon}bar: NOT FOUND")

print("-" * 60)
print(f"\n✓ Total: 20 datasets")
print(f"✓ Ready for Step 5: Feature Analysis & Model Training")

STEP 4 COMPLETE: DATASET MATRIX

Datasets directory: /media/przem/linux_data/RiskYieldMM (Copy)/data/datasets
------------------------------------------------------------
✓ direction_1bar: 168 base + 2 interactions, 5,437 valid
✓ direction_3bar: 171 base + 5 interactions, 5,435 valid
✓ direction_6bar: 170 base + 4 interactions, 5,432 valid
✓ direction_12bar: 171 base + 5 interactions, 5,426 valid
✓ returns_1bar: 171 base + 5 interactions, 5,437 valid
✓ returns_3bar: 171 base + 5 interactions, 5,435 valid
✓ returns_6bar: 171 base + 5 interactions, 5,432 valid
✓ returns_12bar: 171 base + 5 interactions, 5,426 valid
✓ volatility_1bar: 171 base + 5 interactions, 5,437 valid
✓ volatility_3bar: 171 base + 5 interactions, 5,435 valid
✓ volatility_6bar: 171 base + 5 interactions, 5,432 valid
✓ volatility_12bar: 166 base + 0 interactions, 5,426 valid
✓ vol_regime_1bar: 171 base + 5 interactions, 5,438 valid
✓ vol_regime_3bar: 171 base + 5 interactions, 5,438 valid
✓ vol_regime_6bar: 171 base + 

# Step 5: Feature Analysis (IC/ICIR)

Compute Information Coefficient (IC) and IC Information Ratio (ICIR) for all features.

**Metrics computed:**
- **IC** (Spearman correlation): Feature's predictive power
- **ICIR** = mean(IC) / std(IC): Stability of predictive power
- **Hit Rate**: Directional accuracy
- **FDR-corrected significance**: Multiple testing correction

**Multi-horizon analysis** uses non-overlapping samples for horizons > 1 bar to avoid autocorrelation bias.

**Output:** 
- `data/analysis/results/ic_multi_horizon.csv` — All features across horizons
- `data/analysis/results/ic_horizon_summary.csv` — Best horizon per feature

In [12]:
# Step 5: Feature Analysis (IC/ICIR) across all horizons
from scripts.analysis.run import cmd_features

# Compute multi-horizon IC analysis
# - IC for each feature against y_forward_return_{1,3,6,12}
# - Uses non-overlapping samples for horizons > 1 (to avoid autocorrelation)
# - FDR correction for multiple testing
# - Identifies best horizon per feature

print("Running multi-horizon IC analysis...")
print("This computes Information Coefficient for each feature against all forward return horizons\n")

ic_results = cmd_features(plot=False)  # Set plot=True to generate visualizations

Running multi-horizon IC analysis...
This computes Information Coefficient for each feature against all forward return horizons

FEATURE ANALYSIS (Multi-Horizon)

--- Multi-Horizon IC Analysis ---
Using non-overlapping samples for horizons > 1 bar
Horizons: {1: '8h', 3: '24h', 6: '48h', 12: '96h'}

Top 20 features by |best_ic|:
                        feature      domain  ic_1bar  ic_3bar  ic_6bar  ic_12bar  best_horizon
       V_hurstExponent_63_rat_N  Volatility  +0.0259  +0.0640  +0.1069   +0.1248            12
          V_volMomentum_6_pct_N  Volatility  +0.0032  +0.0104  -0.0062   -0.1044            12
      V_hurstExponent_126_rat_N  Volatility  +0.0232  +0.0312  +0.0570   +0.0928            12
             M_T_V_adx_12_bnd_N  Volatility  +0.0303  +0.0648  +0.0919   +0.0545             6
    S_M_longShortChange_3_pct_N   Sentiment  -0.0238  -0.0246  -0.0527   -0.0880            12
         S_longShortRatio_rat_N   Sentiment  -0.0208  -0.0192  -0.0370   -0.0875            12
     

In [13]:
# Show top features by IC
import pandas as pd

print("=" * 60)
print("TOP FEATURES BY |IC|")
print("=" * 60)

# Display top 20 features with their IC across horizons
if ic_results is not None:
    display_cols = ['feature', 'domain', 'ic_1bar', 'ic_3bar', 'ic_6bar', 'ic_12bar', 'best_horizon']
    top_20 = ic_results[display_cols].head(20)
    
    # Format IC values
    for col in ['ic_1bar', 'ic_3bar', 'ic_6bar', 'ic_12bar']:
        top_20[col] = top_20[col].apply(lambda x: f"{x:+.4f}" if pd.notna(x) else "")
    
    print(top_20.to_string(index=False))
    
    # Domain summary
    print("\n" + "-" * 60)
    print("TOP FEATURES BY DOMAIN")
    print("-" * 60)
    domain_counts = ic_results.groupby('domain').agg({
        'feature': 'count',
        'ic_1bar': lambda x: x.abs().mean()
    }).rename(columns={'feature': 'count', 'ic_1bar': 'mean_|IC|'})
    domain_counts = domain_counts.sort_values('mean_|IC|', ascending=False)
    print(domain_counts.to_string())

TOP FEATURES BY |IC|
                        feature      domain ic_1bar ic_3bar ic_6bar ic_12bar  best_horizon
       V_hurstExponent_63_rat_N  Volatility +0.0259 +0.0640 +0.1069  +0.1248            12
          V_volMomentum_6_pct_N  Volatility +0.0032 +0.0104 -0.0062  -0.1044            12
      V_hurstExponent_126_rat_N  Volatility +0.0232 +0.0312 +0.0570  +0.0928            12
             M_T_V_adx_12_bnd_N  Volatility +0.0303 +0.0648 +0.0919  +0.0545             6
    S_M_longShortChange_3_pct_N   Sentiment -0.0238 -0.0246 -0.0527  -0.0880            12
         S_longShortRatio_rat_N   Sentiment -0.0208 -0.0192 -0.0370  -0.0875            12
              M_T_V_adx_6_bnd_N  Volatility +0.0114 +0.0307 +0.0859  +0.0019             6
          C_N_lowerShadow_bnd_N Candlestick -0.0120 +0.0043 -0.0314  -0.0834            12
 F_I_M_fundingMaDiff_7_21_pct_N     Funding -0.0128 -0.0067 -0.0547  -0.0829            12
   S_N_longShortZscore_21_zsc_N   Sentiment -0.0240 -0.0307 -0.0559  

# Step 6: Feature Importance (MDI + MDA)

Two complementary importance methods:

**MDI (Mean Decrease Impurity):**
- Fast, from LightGBM's built-in feature importances
- Can be biased toward high-cardinality features

**MDA (Mean Decrease Accuracy / Permutation Importance):**
- Out-of-sample, model-agnostic
- Based on Lopez de Prado, AFML Ch.8
- More reliable but slower

**Output:**
- `data/analysis/results/feature_importance.csv` (MDI)
- `data/analysis/results/mda_importance.csv` (MDA)

In [14]:
# Step 6a: MDI (Mean Decrease Impurity) - Fast importance from LightGBM
from scripts.analysis.run import cmd_importance

print("Computing MDI feature importance...")
mdi_results = cmd_importance(plot=False)

print("\n" + "=" * 60)
print("TOP 20 FEATURES BY MDI IMPORTANCE")  
print("=" * 60)
print(mdi_results.head(20).to_string(index=False))

Computing MDI feature importance...
FEATURE IMPORTANCE (MDI)
Training LightGBM for importance...

Top 20 features:
                       feature  importance       domain
   S_M_longShortChange_3_pct_N  560.879350    Sentiment
     V_hurstExponent_126_rat_N  523.415251   Volatility
       L_M_N_volumeRoc_3_pct_N  520.596919       Volume
            N_P_V_pctB_3_bnd_N  461.508710   Volatility
          M_N_rocAccel_3_dif_N  425.954296     Momentum
    M_P_D_markReturnDiff_pct_N  405.443805     Momentum
               V_skew_42_rat_N  402.314452   Volatility
     N_P_L_vwapDeviation_pct_N  401.066086        Other
      V_hurstExponent_63_rat_N  400.290576   Volatility
         C_N_upperShadow_bnd_N  390.095798  Candlestick
     L_M_N_S_oiPctChange_pct_N  389.107410 OpenInterest
             L_M_S_cmf_6_bnd_N  387.287277       Volume
       L_M_N_volumeRoc_6_pct_N  382.165964       Volume
         C_N_lowerShadow_bnd_N  377.490080  Candlestick
         L_N_S_oiRatio_6_rat_N  369.279749 Op

In [15]:
# Step 6b: MDA (Permutation Importance) - Out-of-sample, model-agnostic
from scripts.analysis.run import cmd_mda

print("Computing MDA feature importance (permutation-based)...")
print("This is more reliable but slower than MDI\n")

mda_results = cmd_mda(n_repeats=5)  # 5 repeats for speed, use 10 for production

print("\n" + "=" * 60)
print("TOP 20 FEATURES BY MDA IMPORTANCE")
print("=" * 60)
print(mda_results.head(20).to_string(index=False))

Computing MDA feature importance (permutation-based)...
This is more reliable but slower than MDI

MDA (Permutation Importance)
Based on: Lopez de Prado, AFML Ch.8
Method: Out-of-sample, model-agnostic

Data: 4,349 train, 1,088 validation samples
Features: 166

Training LightGBM...

Computing permutation importance (n_repeats=5)...

Top 30 features by MDA:
                        feature  importance_mean  importance_std       domain
   S_N_longShortZscore_63_zsc_N         0.006431        0.001339    Sentiment
N_V_bollingerBandwidth_42_pct_N         0.006397        0.004679   Volatility
      V_hurstExponent_126_rat_N         0.005509        0.002190   Volatility
   S_N_longShortZscore_21_zsc_N         0.004728        0.000970    Sentiment
            V_volOfVol_21_pct_N         0.004560        0.000626   Volatility
            M_V_calmar_21_rat_N         0.004393        0.002984   Volatility
           M_T_V_diDiff_6_bnd_N         0.004080        0.000754   Volatility
            M_V_s

# Step 7: Cross-Validation

Time-series cross-validation with proper temporal separation.

**PurgedKFold CV** (from Lopez de Prado, AFML Ch.7):
- **Purge gap** (21 bars): Removes training samples that could have features using test period data
- **Embargo gap** (12 bars): Removes training samples whose targets overlap with test period

**Models trained:**
- CatBoost Classifier (direction)
- LightGBM Classifier (direction)
- CatBoost Regressor (volatility)

**Output:** `data/analysis/results/cv_results.csv`

In [16]:
# Step 7: Cross-Validation with PurgedKFold
from scripts.analysis.run import cmd_cv

print("Running PurgedKFold Cross-Validation...")
print("  purge_gap=21 bars (prevents feature look-ahead)")
print("  embargo_gap=12 bars (prevents target overlap)\n")

cv_results = cmd_cv(use_purged=True)

# Summary
print("\n" + "=" * 60)
print("CROSS-VALIDATION SUMMARY")
print("=" * 60)
for model_name, cv_result in cv_results.items():
    print(f"  {model_name:10s}: AUC = {cv_result.mean_auc:.4f} ± {cv_result.std_auc:.4f}")

Running PurgedKFold Cross-Validation...
  purge_gap=21 bars (prevents feature look-ahead)
  embargo_gap=12 bars (prevents target overlap)

CROSS-VALIDATION (PurgedKFold)
purge_gap=21, embargo_gap=12

--- Direction Model CV ---

Direction CV Summary:
  catboost: AUC=0.5317 ± 0.0160
  lightgbm: AUC=0.5086 ± 0.0229
  ensemble: AUC=0.5108 ± 0.0248

--- Volatility Model CV ---
  Volatility: RMSE=0.011826 ± 0.003191

✓ Saved: /media/przem/linux_data/RiskYieldMM (Copy)/data/analysis/results/cv_results.csv

CROSS-VALIDATION SUMMARY
  catboost  : AUC = 0.5317 ± 0.0160
  lightgbm  : AUC = 0.5086 ± 0.0229
  ensemble  : AUC = 0.5108 ± 0.0248


# Pipeline Complete ✓

**Summary of outputs:**

| Step | Output | Location |
|------|--------|----------|
| 1a | Raw merged data | `data/merged_8h_raw.parquet` |
| 1b | Computed features | `data/features_8h.parquet` |
| 2 | Analysis dataset + targets | `data/analysis_8h.parquet` |
| 3 | Optimized features | `data/features_8h_optimized_{target}_{horizon}bar.parquet` (20 files) |
| 4 | Final datasets | `data/datasets/{target}_{horizon}bar.parquet` (20 files) |
| 5 | IC/ICIR analysis | `data/analysis/results/ic_*.csv` |
| 6 | Feature importance | `data/analysis/results/{feature_importance,mda_importance}.csv` |
| 7 | CV results | `data/analysis/results/cv_results.csv` |

**Next steps:**
- Walk-forward backtest: `cmd_backtest()`
- Full analysis: `cmd_all()`

In [17]:
# Final pipeline verification
from pathlib import Path
import os

print("=" * 70)
print("PIPELINE VERIFICATION")
print("=" * 70)

checks = []

# Check Step 1: Raw and features
raw_file = PROJECT_ROOT / "data" / "merged_8h_raw.parquet"
features_file = PROJECT_ROOT / "data" / "features_8h.parquet"
checks.append(("Step 1a: merged_8h_raw.parquet", raw_file.exists()))
checks.append(("Step 1b: features_8h.parquet", features_file.exists()))

# Check Step 2: Analysis dataset
analysis_file = PROJECT_ROOT / "data" / "analysis_8h.parquet"
checks.append(("Step 2: analysis_8h.parquet", analysis_file.exists()))

# Check Step 3: Optimized features (20 files)
opt_count = len(list(PROJECT_ROOT.glob("data/features_8h_optimized_*.parquet")))
checks.append((f"Step 3: Optimized features ({opt_count}/20)", opt_count == 20))

# Check Step 4: Final datasets (20 files)
datasets_dir = PROJECT_ROOT / "data" / "datasets"
dataset_count = len(list(datasets_dir.glob("*.parquet"))) if datasets_dir.exists() else 0
checks.append((f"Step 4: Final datasets ({dataset_count}/20)", dataset_count == 20))

# Check Step 5-7: Results
results_dir = PROJECT_ROOT / "data" / "analysis" / "results"
ic_file = results_dir / "ic_multi_horizon.csv"
mdi_file = results_dir / "feature_importance.csv"
mda_file = results_dir / "mda_importance.csv"
cv_file = results_dir / "cv_results.csv"

checks.append(("Step 5: IC analysis", ic_file.exists()))
checks.append(("Step 6a: MDI importance", mdi_file.exists()))
checks.append(("Step 6b: MDA importance", mda_file.exists()))
checks.append(("Step 7: CV results", cv_file.exists()))

# Print results
print()
all_passed = True
for name, passed in checks:
    status = "✓" if passed else "✗"
    print(f"  {status} {name}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("═" * 70)
    print("ALL PIPELINE STEPS COMPLETE ✓")
    print("═" * 70)
else:
    print("Some steps incomplete - run the missing cells above")

PIPELINE VERIFICATION

  ✓ Step 1a: merged_8h_raw.parquet
  ✓ Step 1b: features_8h.parquet
  ✓ Step 2: analysis_8h.parquet
  ✗ Step 3: Optimized features (25/20)
  ✓ Step 4: Final datasets (20/20)
  ✓ Step 5: IC analysis
  ✓ Step 6a: MDI importance
  ✓ Step 6b: MDA importance
  ✓ Step 7: CV results

Some steps incomplete - run the missing cells above


# Step 8: L1 Precomputation

Precompute L1 helper features for all 20 configs (5 targets × 4 horizons).

**Walk-forward config:**
- `backtest_rows`: 500 iterations
- `L1_warmup`: 500 bars minimum history before first prediction
- `L1_window`: 500 bars training window

**What it does per iteration:**
1. `fit(X[train_window])` — Fit L1 ensemble on training data
2. `transform(X[pred_row])` — Generate helper features for prediction row
3. Save `{timestamp}.parquet` with helper features

**Resume capability:**
- Detects if new data was fetched → regenerates to include new rows
- Detects if `backtest_rows` changed → regenerates with new count
- Set `force=True` to regenerate everything from scratch

**Storage:** `data/precomputed/{config_name}/{YYYY-MM-DD_HHh}.parquet`

**Benefit:** After precomputation, backtests load pre-saved helper features instead of recomputing L1 each iteration.

In [ ]:
# Step 8: Precompute L1 helper features for all 20 configs
#
# This ONLY computes L1 features. Later layers use these precomputed outputs.
# NOW INCLUDES NEW HELPERS: EVT POT, OU, BOCPD, EGARCH
# RUST BACKENDS: 7 helpers use high-performance Rust implementations

import json
import time
from dataclasses import asdict

import pandas as pd

from scripts.target_models.validation.l1_precompute import (
    L1PrecomputeConfig,
    get_all_config_names,
    load_iteration_index,
    precompute_l1_for_config,
)
from scripts.target_models.validation.regenerate_all_precomputes import (
    _filtered_end_timestamp_and_last_idx,
    verify_config_finishes_at_dataset_end,
)

# ============================================================================
# RUST BACKEND STATUS
# ============================================================================
from scripts.target_models.helpers.cusum import HAS_RUST as CUSUM_RUST
from scripts.target_models.helpers.kalman import HAS_RUST as KALMAN_RUST
from scripts.target_models.helpers.garch import HAS_RUST as GARCH_RUST
from scripts.target_models.helpers.evt_pot import HAS_RUST as EVT_RUST
from scripts.target_models.helpers.ou import HAS_RUST as OU_RUST
from scripts.target_models.helpers.bocpd import HAS_RUST as BOCPD_RUST
from scripts.target_models.helpers.egarch import HAS_RUST as EGARCH_RUST

print("=" * 60)
print("RUST BACKEND STATUS")
print("=" * 60)
rust_status = {
    "CUSUM": CUSUM_RUST,
    "Kalman": KALMAN_RUST,
    "GARCH": GARCH_RUST,
    "EVT": EVT_RUST,
    "OU": OU_RUST,
    "BOCPD": BOCPD_RUST,
    "EGARCH": EGARCH_RUST,
}
for name, has_rust in rust_status.items():
    status = "✓ Rust" if has_rust else "○ Python"
    speedup = {"CUSUM": "156x", "Kalman": "487x", "GARCH": "224x", 
               "EVT": "25x", "OU": "30x", "BOCPD": "20x", "EGARCH": "22x"}
    print(f"  {name:8s}: {status} ({speedup.get(name, '')} speedup)" if has_rust else f"  {name:8s}: {status}")

rust_count = sum(rust_status.values())
print(f"\n  {rust_count}/7 helpers using Rust backends")
print("  HMM4, HMM5, IsolationForest use optimized Python (hmmlearn/sklearn)")

# ============================================================================
# CONFIGURATION — SET YOUR L1 ITERATION COUNT HERE
# ============================================================================
L1_ROWS = 2500  # <-- How many L1 iterations to precompute (change this!)

# NEW: Include all 10 helpers (6 original + 4 new)
HELPERS = ["if", "cusum", "garch", "hmm4", "hmm5", "kalman", "evt", "ou", "bocpd", "egarch"]
print(f"\nL1 Helpers enabled: {HELPERS}")

# DISABLE feature selection - keep ALL features from helpers
ENABLE_BOOSTING = False  # Set True to enable ICIR feature selection

cfg = L1PrecomputeConfig(
    data_dir=PROJECT_ROOT / "data" / "datasets",
    output_dir=PROJECT_ROOT / "data" / "precomputed",
    backtest_rows=L1_ROWS,
    random_state=42,
    helpers=HELPERS,  # Pass new helpers list
    enable_boosting=ENABLE_BOOSTING,  # False = keep all features
)

print(f"Feature selection: {'ENABLED (ICIR filtering)' if ENABLE_BOOSTING else 'DISABLED (keep all features)'}")

configs = get_all_config_names()
force = True  # Force regenerate to include new helpers
align_global_end = True
no_verify = False

# ============================================================================
# Find global end timestamp (all configs truncated to same end)
# ============================================================================
global_end: pd.Timestamp | None = None
if align_global_end:
    ends = [_filtered_end_timestamp_and_last_idx(name, cfg)[0] for name in configs]
    global_end = min(ends)

results: list[dict] = []
end_timestamps: dict[str, str] = {}

print("\n" + "=" * 60)
print("L1 PRECOMPUTATION (WITH RUST ACCELERATION)")
print("=" * 60)
print(f"\n  L1 iterations to compute: {L1_ROWS}")
print(f"  Helpers: {len(HELPERS)} ({', '.join(HELPERS)})")
print(f"  Configs: {len(configs)} (5 targets × 4 horizons)")
print(f"  Output: {cfg.output_dir}")
if global_end is not None:
    print(f"  Global end: {global_end.strftime('%Y-%m-%d %H:%M')}")
print(f"  Force regenerate: {force}")
print()

# ============================================================================
# Run L1 precomputation
# ============================================================================
total_t0 = time.time()
skipped_count = 0
regenerated_count = 0

for i, config_name in enumerate(configs, start=1):
    print(f"[{i}/{len(configs)}] {config_name}")
    
    metadata_file = cfg.metadata_path(config_name)
    action = "compute"
    
    if metadata_file.exists() and not force:
        with open(metadata_file) as f:
            existing_meta = json.load(f)
        existing_rows = existing_meta.get("total_iterations", 0)
        existing_end = existing_meta.get("last_pred_timestamp", "")
        
        current_end_ts, _ = _filtered_end_timestamp_and_last_idx(config_name, cfg)
        current_end = current_end_ts.isoformat() if global_end is None else global_end.isoformat()
        
        if existing_rows == L1_ROWS:
            if existing_end >= current_end:
                print(f"  ⏭️  Done ({existing_rows} L1 iterations)")
                existing_meta["skipped"] = True
                results.append(existing_meta)
                skipped_count += 1
                continue
            else:
                print(f"  📈 NEW DATA: L1 ends {existing_end[:10]}, data ends {current_end[:10]}")
                action = "regenerate_new_data"
                regenerated_count += 1
        elif existing_rows < L1_ROWS:
            print(f"  🔄 Have {existing_rows} L1 rows, need {L1_ROWS}")
            action = "regenerate_more_rows"
            regenerated_count += 1
        else:
            print(f"  ⏭️  Have {existing_rows} L1 rows (≥ {L1_ROWS})")
            existing_meta["skipped"] = True
            results.append(existing_meta)
            skipped_count += 1
            continue
    
    t0 = time.time()
    meta = precompute_l1_for_config(
        config_name,
        cfg=cfg,
        verbose=True,
        force=True,
        max_timestamp=global_end,
    )
    dt = time.time() - t0

    meta_dict = asdict(meta)
    meta_dict["elapsed_sec"] = round(dt, 2)
    meta_dict["action"] = action

    if meta.total_iterations != L1_ROWS:
        raise RuntimeError(
            f"{config_name}: got {meta.total_iterations} L1 iterations, expected {L1_ROWS}"
        )

    if not no_verify:
        check = verify_config_finishes_at_dataset_end(
            config_name, cfg, max_timestamp=global_end
        )
        meta_dict["end_alignment"] = check
        if not check["ok"]:
            raise RuntimeError(f"End alignment failed: {json.dumps(check, indent=2)}")
        end_timestamps[config_name] = check["data_last_timestamp"]

    results.append(meta_dict)
    print(f"  ✓ {meta.total_iterations} L1 iterations in {dt:.1f}s")

total_dt = time.time() - total_t0

# ============================================================================
# Verify all configs aligned
# ============================================================================
if not no_verify and end_timestamps:
    unique_ends = sorted(set(end_timestamps.values()))
    if len(unique_ends) != 1:
        print(f"\n⚠️  WARNING: {len(unique_ends)} different end timestamps (expected 1)")

# ============================================================================
# Save summary
# ============================================================================
summary_path = cfg.output_dir / "precompute_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with open(summary_path, "w") as f:
    json.dump(
        {
            "l1_iterations": L1_ROWS,
            "helpers": HELPERS,
            "rust_backends": rust_count,
            "output_dir": str(cfg.output_dir),
            "data_dir": str(cfg.data_dir),
            "force": bool(force),
            "total_elapsed_sec": round(total_dt, 2),
            "configs": results,
        },
        f,
        indent=2,
    )

print(f"\n{'=' * 60}")
print(f"L1 PRECOMPUTATION COMPLETE")
print(f"{'=' * 60}")
print(f"  L1 iterations: {L1_ROWS}")
print(f"  Helpers: {len(HELPERS)} (7 Rust + 3 Python)")
print(f"  Skipped: {skipped_count}")
print(f"  Computed: {regenerated_count}")
print(f"  Time: {total_dt / 60:.1f} min")
print(f"\n  Output: {cfg.output_dir}")
print(f"  Summary: {summary_path}")

RUST BACKEND STATUS
  CUSUM   : ✓ Rust (156x speedup)
  Kalman  : ✓ Rust (487x speedup)
  GARCH   : ✓ Rust (224x speedup)
  EVT     : ✓ Rust (25x speedup)
  OU      : ✓ Rust (30x speedup)
  BOCPD   : ✓ Rust (20x speedup)
  EGARCH  : ✓ Rust (22x speedup)

  7/7 helpers using Rust backends
  HMM4, HMM5, IsolationForest use optimized Python (hmmlearn/sklearn)

L1 Helpers enabled: ['if', 'cusum', 'garch', 'hmm4', 'hmm5', 'kalman', 'evt', 'ou', 'bocpd', 'egarch']

L1 PRECOMPUTATION (WITH RUST ACCELERATION)

  L1 iterations to compute: 2500
  Helpers: 10 (if, cusum, garch, hmm4, hmm5, kalman, evt, ou, bocpd, egarch)
  Configs: 20 (5 targets × 4 horizons)
  Output: /media/przem/linux_data/RiskYieldMM (Copy)/data/precomputed
  Global end: 2025-12-14 08:00
  Force regenerate: True

[1/20] returns_1bar

L1 Precomputation: returns_1bar
Dropped 1002 NaN rows (5438 → 4436)
Truncated to timestamp <= 2025-12-14T08:00:00+00:00
Loaded: 4436 rows, 171 base features
Target column: y_returns

  L1 config:

KeyboardInterrupt: 

In [ ]:
# Verify L1 precomputation
from pathlib import Path
import json

precomputed_dir = PROJECT_ROOT / "data" / "precomputed"

# Re-import in case running standalone
from scripts.target_models.validation.l1_precompute import get_all_config_names
config_names = get_all_config_names()

print("=" * 60)
print("L1 PRECOMPUTATION VERIFICATION")
print("=" * 60)

# Check summary file for helpers and Rust info
summary_path = precomputed_dir / "precompute_summary.json"
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    helpers = summary.get("helpers", [])
    rust_count = summary.get("rust_backends", 0)
    print(f"\n  Helpers: {len(helpers)} ({', '.join(helpers)})")
    print(f"  Rust backends: {rust_count}/7")
    print(f"  Total time: {summary.get('total_elapsed_sec', 0) / 60:.1f} min")

valid_configs = []
missing_configs = []

for config_name in config_names:
    metadata_path = precomputed_dir / config_name / "metadata.json"
    
    if metadata_path.exists():
        with open(metadata_path) as f:
            meta = json.load(f)
        n_iters = meta["total_iterations"]
        date_range = f"{meta['first_pred_timestamp'][:10]} to {meta['last_pred_timestamp'][:10]}"
        valid_configs.append((config_name, n_iters, date_range))
    else:
        missing_configs.append(config_name)

print(f"\n✓ Precomputed configs: {len(valid_configs)}/{len(config_names)}")
for name, n_iters, date_range in valid_configs[:5]:
    print(f"  {name}: {n_iters} iterations ({date_range})")
if len(valid_configs) > 5:
    print(f"  ... and {len(valid_configs) - 5} more")

if missing_configs:
    print(f"\n✗ Missing configs: {len(missing_configs)}")
    for name in missing_configs[:5]:
        print(f"  {name}")
    if len(missing_configs) > 5:
        print(f"  ... and {len(missing_configs) - 5} more")
else:
    print("\n✓ All 20 configs precomputed and ready for fast backtest")